In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount("/content/drive") # connect Google Colab to a Drive .csv file

# importing the necessary libraries and modules for the random forest model
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# built-in modules
import os
import warnings
import time
warnings.filterwarnings("ignore")

# download dataset via kagglehub
path = kagglehub.dataset_download("shadmanrohan/collisionavoidancechallenge")

print("Path to dataset files:", path)

import sys
print(sys.version)

In [ ]:
os.chdir(path)
os.listdir(path)
df = pd.read_csv('train_data.csv')

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/35_feature_RF5.csv" # saving the path of the CSV file

In [ ]:
# setting the y variable to the risk feature and
# categorizing the y variable into multiple categories
y = df["risk"].values.astype(float)
yCategories = np.empty_like(y, dtype=int)
yCategories[y < -16] = 1                     # low-risk
yCategories[(y >= -16) & (y < -9)] = 2       # medium-risk
yCategories[y >= -9] = 3                     # high-risk

In [ ]:
# setting a new dataframe to read the CSV file
modelChoice = "multi" # can switch depending on what model is being run

# single-feature random forest model
if modelChoice == "single":
    if os.path.exists(SAVE_PATH):
        accuracyDF = pd.read_csv(SAVE_PATH)
        finishedFeatures = set(accuracyDF["feature"].unique())
    else:
        accuracyDF = pd.DataFrame(columns=["feature", "accuracy"])
        finishedFeatures = set()
elif modelChoice == "multi":
    # multi-feature random forest model
    if os.path.exists(SAVE_PATH):
        accuracyDF = pd.read_csv(SAVE_PATH)
        finishedTrials = set(accuracyDF["trial"])
        print(f"Resuming — {len(finishedTrials)} trials already done")
    else:
        accuracyDF = pd.DataFrame(columns=["trial", "accuracy"])
        finishedTrials = set()

In [ ]:
# ML model programming
if modelChoice == "single":
  # single-feature random forest model code
  allColumns = [c for c in df.columns if c == "t_sigma_r" or c == "c_sigma_r" or c == "t_sigma_t" or c == "c_sigma_t" or \
                c == "t_sigma_n" or c=="c_sigma_n" or c=="t_sigma_rdot" or c=="c_sigma_rdot" or c=="t_sigma_tdot" or \
                c=="c_sigma_tdot" or c=="t_sigma_ndot" or c=="c_sigma_ndot" or c=="F10" or c=="F3M" or c=="SSN" or \
                c=="AP"]
  for feature in allColumns:
      if feature in finishedFeatures:
          continue  # skip already finished features
      print(f"Started: {feature}")
      X = df[feature].values.reshape(-1, 1)
      accuracies = []
      for i in range(10):
          startTime = time.time()
          X_train, X_test, y_train, y_test = train_test_split(X, yCategories, test_size=0.2, random_state=i)
          model = RandomForestClassifier(n_estimators=25,random_state=i,max_depth=75)
          # fitting, predicting, and calculating accuracy scores based on X_train and Y_train
          model.fit(X_train, y_train)
          y_pred = model.predict(X_test)
          acc = accuracy_score(y_test, y_pred)
          accuracies.append(acc)
      featureDF = pd.DataFrame({
          "feature": [feature] * 10,
          "accuracy": accuracies
      })
      accuracyDF = pd.concat([accuracyDF, featureDF], ignore_index=True)
      accuracyDF.to_csv(SAVE_PATH, index=False)
      endTime = time.time()
      print(f"Completed: {feature}")
      print(f"Elapsed: {endTime - startTime:.8f}")

elif modelChoice == "multi":

    # 20-feature random forest model code
    featureSetName = "top35_model"
    selectedColumns = [ # the full list may be cut off; it includes the selected top 35 features
        c for c in df.columns if c == "c_j2k_inc" or c== "c_cd_area_over_mass" or c== "azimuth" or c=="c_sedr" or c=="c_cr_area_over_mass" or c== "relative_velocity_n" or c=="relative_velocity_t" or c=="c_h_per" or c =="c_j2k_sma" or c== "t_j2k_inc" or c=="c_h_apo" or c=="t_j2k_sma" or c=="c_j2k_ecc" or c=="elevation" or c=="c_sigma_ndot" or c=="c_sigma_n" or c =="relative_speed" or c=="geocentric_latitude" or c=="c_sigma_tdot" or c=="t_h_per" or c=="miss_distance" or c=="c_cndot_n" or c=="c_time_lastob_start" or c=="c_time_lastob_end" or c=="c_actual_od_span" or c=="c_weighted_rms" or c=="c_recommended_od_span" or c=="t_h_apo" or c=="t_rcs_estimate" or c=="c_sigma_r" or c=="c_rcs_estimate" or c=="relative_velocity_r" or c=="c_obs_available" or c=="c_obs_used" or c=="c_sigma_rdot" or c=="c_sigma_t" # using top 20 features from the single-feature analysis + miss_distance
    ]

    # check if results file already exists
    if os.path.exists(SAVE_PATH):
        accuracyDF20 = pd.read_csv(SAVE_PATH)
    else:
        accuracyDF20 = pd.DataFrame(columns=["model", "trial", "accuracy"])

    # skip if already completed
    if featureSetName in accuracyDF20["model"].values:
        print("20-feature model already completed. Skip")
    else:
        print(f"Starting ({featureSetName})")
        X = df[selectedColumns].values
        accuraciesTrain = []
        accuraciesTest = []
        for i in range(10): # begin 10 trials for model
            startTime = time.time()
            X_train, X_test, y_train, y_test = train_test_split(X, yCategories, test_size=0.2, random_state=i)
            model = RandomForestClassifier(n_estimators=12,random_state=i,n_jobs=-1)
            model.fit(X_train, y_train)
            y_pred_train = model.predict(X_train)
            y_pred_test = model.predict(X_test)
            accTrain = accuracy_score(y_train, y_pred_train)
            accTest = accuracy_score(y_test, y_pred_test)
            accuraciesTrain.append(accTrain)


            # appending test accuracies
            accuraciesTest.append(accTest)
            trialDF = pd.DataFrame({
                "model": [featureSetName],
                "trial": [i],
                "accuracyTrain": [accTrain],
                "accuracyTest": [accTest]
            })
            accuracyDF20 = pd.concat([accuracyDF20, trialDF], ignore_index=True)
            accuracyDF20.to_csv(SAVE_PATH, index=False)
        endTime = time.time()
        print(f"finished ({featureSetName})")
        print("Mean training accuracy:", np.mean(accuraciesTrain))
        print("Mean testing accuracy:", np.mean(accuraciesTest))
        elapsed = endTime - startTime
        print(f"{elapsed:.4f}")